# Bài 9 — Sản phẩm cuối: Traffic Monitor hoàn chỉnh

**Mục tiêu:** Ghép Bài 1–8 thành sản phẩm chạy được, cấu trúc code chuẩn, cửa sổ giám sát live + bảng thống kê realtime, xuất video + báo cáo JSON.

Code bám sát đúng cấu trúc README: 5 file `config.py`, `display.py`, `transformer.py`, `monitor.py`, `main.py`.

> Chỉ 1 chỗ khác README gốc: dùng `self.class_names = self.model.names` tra theo `class_id` thay vì `detections.data["class_name"]`, vì key này bị ByteTrack loại bỏ sau khi track (lỗi `KeyError` thực tế đã gặp ở Bài 5-8).

## 0. Chuẩn bị (asset video + display.py dùng chung)

In [ ]:
!pip install -q supervision ultralytics "supervision[assets]"

In [ ]:
from supervision.assets import download_assets, VideoAssets

download_assets(VideoAssets.VEHICLES)
print(VideoAssets.VEHICLES.value)  # "vehicles.mp4"

In [ ]:
%%writefile display.py
# display.py — hàm hiển thị dùng chung cho toàn giáo trình
import cv2

WINDOW_NAME = "Supervision - Live"
MAX_DISPLAY_WIDTH = 1280   # thu nhỏ frame cho vừa màn hình (chỉ để XEM, không ảnh hưởng xử lý)


def show_frame(frame, window_name: str = WINDOW_NAME, wait: int = 1) -> bool:
    """Hiện frame lên cửa sổ. Trả về False nếu người dùng bấm Q/ESC (muốn thoát).

    wait=1  -> dùng cho video (hiện liên tục, không chặn)
    wait=0  -> dùng cho ảnh tĩnh (dừng lại chờ bấm phím bất kỳ)
    """
    h, w = frame.shape[:2]
    if w > MAX_DISPLAY_WIDTH:                      # thu nhỏ để vừa màn hình
        scale = MAX_DISPLAY_WIDTH / w
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

    cv2.imshow(window_name, frame)
    key = cv2.waitKey(wait) & 0xFF
    if key in (ord("q"), ord("Q"), 27):            # Q hoặc ESC -> thoát
        return False
    return True


def close_windows():
    cv2.destroyAllWindows()

## `config.py` — mọi tham số chỉnh ở đây

In [ ]:
%%writefile config.py
import numpy as np

SOURCE_VIDEO = "vehicles.mp4"
TARGET_VIDEO = "output_final.mp4"
REPORT_PATH = "report.json"

MODEL_NAME = "yolov8n.pt"       # đổi yolov8s/m nếu có GPU
CONF_THRESHOLD = 0.3
VEHICLE_CLASSES = [2, 3, 5, 7]  # car, motorcycle, bus, truck
SPEED_LIMIT_KMH = 80

SHOW_PREVIEW = True             # False nếu chạy trên server không màn hình

# Hiệu chỉnh theo video của em (Bài 6, Bài 8)
LINE_Y_RATIO = 0.5              # vạch đếm ở 50% chiều cao khung hình
PERSPECTIVE_SOURCE = np.array([[1252, 787], [2298, 803], [5039, 2159], [-550, 2159]])
ROAD_WIDTH_M, ROAD_LENGTH_M = 25, 250

## `transformer.py` — ViewTransformer (Bài 8)

In [ ]:
%%writefile transformer.py
import cv2
import numpy as np


class ViewTransformer:
    def __init__(self, source: np.ndarray, target: np.ndarray):
        self.m = cv2.getPerspectiveTransform(
            source.astype(np.float32), target.astype(np.float32))

    def transform_points(self, points: np.ndarray) -> np.ndarray:
        if points.size == 0:
            return points
        reshaped = points.reshape(-1, 1, 2).astype(np.float32)
        return cv2.perspectiveTransform(reshaped, self.m).reshape(-1, 2)

## `monitor.py` — trái tim sản phẩm

In [ ]:
%%writefile monitor.py
import json
from collections import defaultdict, deque

import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO

import config
from display import show_frame, close_windows
from transformer import ViewTransformer


class TrafficMonitor:
    def __init__(self):
        self.model = YOLO(config.MODEL_NAME)
        self.class_names = self.model.names   # {class_id: ten} — thay cho data["class_name"]
        self.video_info = sv.VideoInfo.from_video_path(config.SOURCE_VIDEO)
        fps = self.video_info.fps

        # --- Tracking (Bài 5) ---
        self.tracker = sv.ByteTrack(frame_rate=fps)
        self.smoother = sv.DetectionsSmoother(length=5)

        # --- Đếm qua vạch (Bài 6) ---
        y = int(self.video_info.height * config.LINE_Y_RATIO)
        self.line_zone = sv.LineZone(
            start=sv.Point(0, y),
            end=sv.Point(self.video_info.width, y),
            triggering_anchors=[sv.Position.BOTTOM_CENTER],
            minimum_crossing_threshold=2,
        )
        self.class_counts = defaultdict(int)

        # --- Đo tốc độ (Bài 8) ---
        target = np.array([
            [0, 0], [config.ROAD_WIDTH_M - 1, 0],
            [config.ROAD_WIDTH_M - 1, config.ROAD_LENGTH_M - 1],
            [0, config.ROAD_LENGTH_M - 1],
        ])
        self.view_transformer = ViewTransformer(config.PERSPECTIVE_SOURCE, target)
        self.coordinates = defaultdict(lambda: deque(maxlen=int(fps)))
        self.speed_violations = {}   # tracker_id -> tốc độ vi phạm cao nhất

        # --- Annotators (Bài 2) ---
        thickness = sv.calculate_optimal_line_thickness(self.video_info.resolution_wh)
        text_scale = sv.calculate_optimal_text_scale(self.video_info.resolution_wh)
        self.box_annotator = sv.BoxAnnotator(
            thickness=thickness, color_lookup=sv.ColorLookup.TRACK)
        self.label_annotator = sv.LabelAnnotator(
            text_scale=text_scale, color_lookup=sv.ColorLookup.TRACK)
        self.trace_annotator = sv.TraceAnnotator(
            trace_length=int(fps * 2), thickness=thickness,
            color_lookup=sv.ColorLookup.TRACK)
        self.line_annotator = sv.LineZoneAnnotator(
            thickness=thickness, text_scale=text_scale,
            custom_in_text="Vao", custom_out_text="Ra")

    # ---------- các bước pipeline ----------

    def detect(self, frame):
        results = self.model(frame, verbose=False)[0]
        detections = sv.Detections.from_ultralytics(results)
        mask = (detections.confidence > config.CONF_THRESHOLD) & \
               np.isin(detections.class_id, config.VEHICLE_CLASSES)
        return detections[mask]

    def track(self, detections):
        detections = self.tracker.update_with_detections(detections)
        return self.smoother.update_with_detections(detections)

    def count(self, detections):
        crossed_in, crossed_out = self.line_zone.trigger(detections)
        for cid in detections.class_id[crossed_in]:
            self.class_counts[f"{self.class_names[cid]}_in"] += 1
        for cid in detections.class_id[crossed_out]:
            self.class_counts[f"{self.class_names[cid]}_out"] += 1

    def speeds(self, detections):
        pts = detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        pts = self.view_transformer.transform_points(pts)
        fps = self.video_info.fps

        labels = []
        for tid, cid, (_, y) in zip(
                detections.tracker_id, detections.class_id, pts):
            name = self.class_names[cid]
            self.coordinates[tid].append(y)
            if len(self.coordinates[tid]) < fps / 2:
                labels.append(f"#{tid} {name}")
                continue
            dist = abs(self.coordinates[tid][-1] - self.coordinates[tid][0])
            speed = dist / (len(self.coordinates[tid]) / fps) * 3.6
            tag = " !QUA TOC DO!" if speed > config.SPEED_LIMIT_KMH else ""
            if tag:
                self.speed_violations[int(tid)] = max(
                    self.speed_violations.get(int(tid), 0), int(speed))
            labels.append(f"#{tid} {name} {int(speed)}km/h{tag}")
        return labels

    def draw_dashboard(self, frame):
        """Bảng thống kê realtime góc trên trái."""
        stats = {
            "Tong VAO": self.line_zone.in_count,
            "Tong RA": self.line_zone.out_count,
            "Vi pham toc do": len(self.speed_violations),
            **dict(self.class_counts),
        }
        x, y, line_h = 25, 70, 55
        h = line_h * len(stats) + 30
        overlay = frame.copy()
        cv2.rectangle(overlay, (10, 10), (560, 10 + h), (0, 0, 0), -1)
        frame[:] = cv2.addWeighted(overlay, 0.6, frame, 0.4, 0)
        for i, (k, v) in enumerate(stats.items()):
            color = (0, 0, 255) if "Vi pham" in k and v > 0 else (255, 255, 255)
            cv2.putText(frame, f"{k}: {v}", (x, y + i * line_h),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.4, color, 3)
        return frame

    def annotate(self, frame, detections, labels):
        out = frame.copy()
        out = self.trace_annotator.annotate(out, detections)
        out = self.box_annotator.annotate(out, detections)
        out = self.label_annotator.annotate(out, detections, labels=labels)
        out = self.line_annotator.annotate(out, line_counter=self.line_zone)
        out = self.draw_dashboard(out)
        return out

    # ---------- chạy ----------

    def process_frame(self, frame):
        detections = self.detect(frame)
        detections = self.track(detections)
        self.count(detections)
        labels = self.speeds(detections)
        return self.annotate(frame, detections, labels)

    def run(self):
        with sv.VideoSink(target_path=config.TARGET_VIDEO,
                          video_info=self.video_info) as sink:
            for frame in sv.get_video_frames_generator(config.SOURCE_VIDEO):
                annotated = self.process_frame(frame)
                sink.write_frame(annotated)
                if config.SHOW_PREVIEW and not show_frame(
                        annotated, window_name="Traffic Monitor - Q de thoat"):
                    print("Da dung theo yeu cau nguoi dung.")
                    break
        close_windows()
        self.export_report()

    def export_report(self):
        report = {
            "video": config.SOURCE_VIDEO,
            "total_in": self.line_zone.in_count,
            "total_out": self.line_zone.out_count,
            "by_class": dict(self.class_counts),
            "speed_violations": self.speed_violations,
        }
        with open(config.REPORT_PATH, "w", encoding="utf-8") as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        print(json.dumps(report, ensure_ascii=False, indent=2))

## `main.py` + chạy sản phẩm

In [ ]:
%%writefile main.py
from monitor import TrafficMonitor

if __name__ == "__main__":
    TrafficMonitor().run()

In [ ]:
# Chạy: python main.py (terminal)
# Hoặc chạy trực tiếp trong notebook:
from monitor import TrafficMonitor

TrafficMonitor().run()

## ✅ Checkpoint Bài 9 (tốt nghiệp)

Sản phẩm chạy end-to-end trên video của chính bạn, cửa sổ giám sát đầy đủ thông tin realtime, số đếm sai lệch < 5% so với đếm tay.

⚠️ `config.PERSPECTIVE_SOURCE` đang là tọa độ mẫu — đo lại theo video thật (Bài 6.4) để tốc độ chính xác.